# Reload data

In [1]:
!pip install -U scikit-learn

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import os
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind

from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, precision_recall_curve
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold


In [3]:
import joblib
X_train, X_test, y_train, y_test = joblib.load('data.joblib')

In [4]:
y_train[y_train==0].shape, y_test.shape

((114,), (104,))

In [5]:
from sklearn.ensemble import IsolationForest

def model_training(clf,  X_train, y_train):
    return clf.fit(X_train, y_train)

def model_evaluation(model, X_test, y_test):
    y_hat = model.predict(X_test)

    # Calculate metrics using the optimal threshold
    optimal_accuracy = accuracy_score(y_test, y_hat)
    optimal_balanced_accuracy = balanced_accuracy_score(y_test, y_hat)
    optimal_precision = precision_score(y_test, y_hat)
    optimal_recall = recall_score(y_test, y_hat)
    optimal_f1 = f1_score(y_test, y_hat)

    # Print metrics using the optimal threshold
    print(f"Accuracy: {optimal_accuracy} balanced {optimal_balanced_accuracy}")
    print(f"Precision: {optimal_precision}")
    print(f"Recall: {optimal_recall}")
    print(f"F1 Score: {optimal_f1}")
    print(confusion_matrix(y_test, y_hat))

    return {"balanced_score": optimal_balanced_accuracy, "f1" : optimal_f1}

In [6]:
import numpy as np
X_train_normal = X_train[y_train==1]
X_test_all = np.concatenate((X_train[y_train==0], X_test), axis=0)
y_test_all = np.concatenate((y_train[y_train==0], y_test), axis=0) # -1 to remap 0 to -1
y_test_all[y_test_all==0] = -1

In [7]:
X_train_normal.shape, X_test_all.shape, y_test_all.shape

((301, 18), (218, 18), (218,))

In [8]:
import mlflow 
mlflow.autolog()
mlflow.set_experiment("sound anomaly")

# With autolog() enabled, all model parameters, a model score, and the fitted model are automatically logged.  
with mlflow.start_run():
    model = model_training(IsolationForest(), X_train, None)
    metrics = model_evaluation(model, X_test_all, y_test_all)
    mlflow.log_metrics(metrics)

2026/11/20 17:20:29 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/11/20 17:20:29 WARNING mlflow.utils.autologging_utils: MLflow statsmodels autologging is known to be compatible with 0.14.3 <= statsmodels, but the installed version is 0.14.0. If you encounter errors during autologging, try upgrading / downgrading statsmodels to a compatible version, or try upgrading MLflow.
2026/11/20 17:20:29 INFO mlflow.tracking.fluent: Autologging successfully enabled for statsmodels.
2026/11/20 17:20:29 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/11/20 17:20:29 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.
2026/11/20 17:20:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbit

Accuracy: 0.591743119266055 balanced 0.677536231884058
Precision: 0.47337278106508873
Recall: 1.0
F1 Score: 0.642570281124498
[[49 89]
 [ 0 80]]
🏃 View run whimsical-mouse-703 at: http://mlflow:5000/#/experiments/1/runs/3f9406098f584ff3976c97bdfffe942e
🧪 View experiment at: http://mlflow:5000/#/experiments/1


In [9]:
import mlflow 
mlflow.autolog()

mlflow.set_experiment("sound anomaly")
# With autolog() enabled, all model parameters, a model score, and the fitted model are automatically logged.  
with mlflow.start_run():
    model = model_training(IsolationForest(n_estimators=300, contamination=0.2), X_train, None)
    metrics = model_evaluation(model, X_test_all, y_test_all)
    mlflow.log_metrics(metrics)

2026/11/20 17:20:33 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/11/20 17:20:33 WARNING mlflow.utils.autologging_utils: MLflow statsmodels autologging is known to be compatible with 0.14.3 <= statsmodels, but the installed version is 0.14.0. If you encounter errors during autologging, try upgrading / downgrading statsmodels to a compatible version, or try upgrading MLflow.
2026/11/20 17:20:33 INFO mlflow.tracking.fluent: Autologging successfully enabled for statsmodels.
2026/11/20 17:20:33 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/11/20 17:20:33 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.
2026/11/20 17:20:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbit

Accuracy: 0.6834862385321101 balanced 0.7263586956521739
Precision: 0.5419847328244275
Recall: 0.8875
F1 Score: 0.6729857819905213
[[78 60]
 [ 9 71]]
🏃 View run amazing-fowl-646 at: http://mlflow:5000/#/experiments/1/runs/db49a0defbf0495fabfedf3262c673eb
🧪 View experiment at: http://mlflow:5000/#/experiments/1


# Server the model

   docker compose --profile serving up

In [10]:
import mlflow 
mlflow.autolog()

mlflow.set_experiment("sound anomaly")
# With autolog() enabled, all model parameters, a model score, and the fitted model are automatically logged.  
with mlflow.start_run():
    model = model_training(IsolationForest(n_estimators=10), X_train, None)
    metrics = model_evaluation(model, X_test_all, y_test_all)
    mlflow.sklearn.log_model(sk_model=model,
                             artifact_path="model",
                             registered_model_name="sound-anomaly",
    )

2026/11/20 17:20:36 INFO mlflow.tracking.fluent: Autologging successfully enabled for sklearn.
2026/11/20 17:20:36 WARNING mlflow.utils.autologging_utils: MLflow statsmodels autologging is known to be compatible with 0.14.3 <= statsmodels, but the installed version is 0.14.0. If you encounter errors during autologging, try upgrading / downgrading statsmodels to a compatible version, or try upgrading MLflow.
2026/11/20 17:20:36 INFO mlflow.tracking.fluent: Autologging successfully enabled for statsmodels.
2026/11/20 17:20:36 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/11/20 17:20:36 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.
2026/11/20 17:20:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbit

Accuracy: 0.5321100917431193 balanced 0.6173007246376812
Precision: 0.436046511627907
Recall: 0.9375
F1 Score: 0.5952380952380952
[[41 97]
 [ 5 75]]


Registered model 'sound-anomaly' already exists. Creating a new version of this model...
2026/11/20 17:20:45 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: sound-anomaly, version 4


🏃 View run mysterious-fox-582 at: http://mlflow:5000/#/experiments/1/runs/56405213a8284405bf2d58361a3f135d
🧪 View experiment at: http://mlflow:5000/#/experiments/1


Created version '4' of model 'sound-anomaly'.


## Serve the model now
     docker compose --profile serving up

In [28]:
import requests

# The API endpoint
url = "http://model-server:5001/invocations"

# Data to be sent
data = {"inputs": X_test_all.tolist() }
print(data)

# A POST request to the API
response = requests.post(url, json=data)

# Print the response
print(response.json())

{'inputs': [[-272.9703369140625, 94.83503723144531, -7.248394966125488, 10.892965316772461, -9.773829460144043, 0.015499517321586609, -4.920855522155762, -4.160910606384277, -8.385744094848633, -4.156550407409668, -4.13283634185791, 0.2689649760723114, -0.024867072701454163, 1779.5903477972763, 3721.845047923323, 18.21997917988069, 0.12866600938498401, 8.246133802458644e-05], [-366.322021484375, 116.45255279541016, -1.1370269060134888, 19.921884536743164, 3.344341516494751, 10.143128395080566, 0.22327901422977448, 1.0856317281723022, -2.2151927947998047, 0.7401853203773499, -2.048241138458252, 0.6781898140907288, 1.5204097032546997, 1376.3736964437396, 2964.1823083067093, 17.201322174370585, 0.08148524860223642, 1.3844251952832565e-05], [-338.0333557128906, 93.3583755493164, -4.994763374328613, 20.68578338623047, -8.24730110168457, 10.224724769592285, -3.633014440536499, -2.0421736240386963, 1.4509011507034302, -4.255414962768555, -2.7410221099853516, 1.4592196941375732, 2.621143102645

In [29]:
y_test_all

array([-1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
       -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
       -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
       -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
       -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
       -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
       -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
       -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,
       -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.,  1.,  1.,  1.,
        1.,  1., -1.,  1.,  1.,  1., -1.,  1.,  1.,  1., -1.,  1., -1.,
        1.,  1., -1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1., -1., -1.,
        1.,  1.,  1.,  1.,  1.,  1.,  1., -1.,  1., -1.,  1.,  1.,  1.,
        1., -1., -1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1., -1.,  1.,
        1., -1.,  1.,  1.,  1., -1., -1., -1.,  1.,  1., -1.,  1